In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchvision.transforms import v2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


# Define transformations for the training and validation sets

transform_train = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop((224, 224)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToDtype(torch.float32,scale = True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = v2.Compose([
    v2.ToImage(),
    v2.Resize((256, 256)),
    v2.CenterCrop((224, 224)),
    v2.ToDtype(torch.float32,scale = True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# load oxford iiit pets dataset
train_dataset = datasets.OxfordIIITPet(root = './data',split = 'trainval',transform = transform_train, download = True)
val_dataset = datasets.OxfordIIITPet(root = './data', split = 'test', transform = transform_val, download = True)

train_loader = DataLoader(train_dataset, batch_size = 128, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 128, shuffle = False)


In [2]:
model = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)
input_features = model.fc.in_features
model.fc = nn.Linear(input_features,37)
model = model.to(device)


In [3]:
epochs = 10 
criterion  = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [12]:
model.load_state_dict(torch.load("checkpoint.pth"))
model = model.to(device)   # always move to device after loading


In [13]:
training_loss = []
for epoch in range(epochs):
    model.train()
    trn_loss = 0
    i = 0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        trn_loss += loss.item()
        i += 1
        print(f"Batch {i}, Loss: {loss.item():.4f}")
        
    average_epoch_loss = trn_loss / len(train_loader)
    training_loss.append(average_epoch_loss)
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {average_epoch_loss:.10f}")

Batch 1, Loss: 0.8383
Batch 2, Loss: 0.7754


KeyboardInterrupt: 

In [14]:
torch.save(model.state_dict(), "checkpoint.pth") # model before it reached 2nd epoch 


In [15]:
correct = 0; 
test_len = len(val_dataset)
model.eval()
with torch.no_grad():
    for batch_X,batch_y in val_loader:
        batch_correct_preds = 0
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        y_pred = model(batch_X)
        pred_labels = torch.argmax(y_pred,dim =1)
        batch_correct_preds += (pred_labels== batch_y).sum().item()
        correct += batch_correct_preds


print(correct)
accuracy = correct/test_len
print(f"Validation Accuracy: {accuracy*100:.10f} %")

2571
Validation Accuracy: 70.0735895339 %


In [ ]:
# here an accuracy of 72.91 percent is caqused because we trained the entire model with lr = 0.001 which is a very high learning rate for the pre-trained layers. We can try to freeze the pre-trained layers and only train the final layer with a higher learning rate, or we can use a lower learning rate for the entire model.
# in the next version of notebook well look at discriminative fine tuning where the lr of initial layers is lower than the lr of the final layers.
# we will also look at learning rate schedulers to adjust the learning rate during training.
#